# 1. Introduction Database

## 1.1 Apa itu database

Database: kumpulan data yang disimpan secara terstruktur sehingga bisa diakses, dikelola dan diperbarui secara sistematis.

Database dirancang agar:
* Data konsisten (tidak ada versi ganda yang saling bertentangan)
* Bisa diakses banyak pengguna/aplikasi secara bersamaan
* Bisa di-query dengan cepat meski volumenya besar

## 1.2 Relational Database

Relational Database (RDBMS): menyimpan data dalam bentuk tabel tabel yang saling terhubung melalui key. Ciri khas:
*  Skema (struktur tabel dan tipe data) bersifat ketat dan didefinisikan di awal
* Data dipecah ke banyak tabel kecil yang saling berelasi, bukan satu tabel raksasa
* Berbeda dengan NoSQL (dokumen/key-value) yang jauh lebih fleksibel tapi kurang ketat strukturnya 

## 1.3 Database vs Table, Row vs Column

Database adalah wadah besar yang berisi banyak table. Table sendiri berbentuk grid mirip spreadsheet, tapi dengan aturan tipe data yang ketat per kolom.

* Row = satu record/observasi (misal: satu customer, satu transaksi)
* Column = satu atribut dari record itu (misal: nama, email, tanggal registrasi)

persis seperti DataFrame di pandas — row = index/baris data, column = fitur/variabel.

## 1.4 Primary Key dan Foreign Key

* Primary Key (PK): kolom yang secara unik mengidentifikasi setiap row dalam satu tabel. Tidak boleh duplikat, tidak boleh NULL. Contoh: customer_id di tabel customers.
* Foreign Key (FK): kolom di suatu tabel yang menunjuk ke Primary Key di tabel lain, untuk membangun relasi antar tabel. Contoh: customer_id di tabel orders adalah FK yang merujuk ke PK customer_id di tabel customers.

Dengan FK ini, sistem tahu "order ini milik customer siapa" tanpa harus menyimpan ulang seluruh data customer di setiap baris order — inilah alasan data dipecah ke banyak tabel, bukan digabung jadi satu.

## 1.5 Schema, SQL, dan DBMS

* Schema: blueprint database — mendefinisikan tabel apa saja yang ada, kolom dan tipe datanya, serta relasi antar tabel.
* SQL (Structured Query Language): bahasa standar untuk berkomunikasi dengan relational database — mengambil, menyaring, mengubah, atau menghapus data.
* DBMS (Database Management System): software yang mengelola database itu sendiri (menyimpan data, menjalankan query, menjaga konsistensi). PostgreSQL adalah contoh DBMS; SQL adalah bahasa yang Anda gunakan untuk "berbicara" dengannya.

![database_structure_ecommerce.png](../assets/database_structure_ecommerce.png)

Komponen-komponen utama:

* Tabel customers (biru): menyimpan data customer. Kolom customer_id adalah Primary Key (ditandai 🔑) — setiap customer punya ID unik yang tidak boleh duplikat.
* Tabel categories (hijau): menyimpan kategori produk. category_id adalah PK-nya.
* Tabel products (kuning): menyimpan produk. product_id adalah PK-nya, tapi ada kolom lain bernama category_id yang ditandai 🔗 (Foreign Key) — ini menunjuk kembali ke tabel categories. Artinya: setiap produk harus punya kategori yang valid di tabel categories.
* Tabel orders (teal): menyimpan order dari customer. order_id adalah PK-nya, dan customer_id adalah FK yang menunjuk ke tabel customers — setiap order harus milik customer yang ada.

Relasi antar tabel:

Panah putus-putus menunjukkan relasi one-to-many:
* Satu customer bisa punya banyak order — tetapi satu order hanya milik satu customer.
* Satu kategori bisa punya banyak produk — tetapi satu produk hanya milik satu kategori.

# 2. Implementasi — Memetakan Struktur Database E-commerce

Database yang akan dipakai punya struktur seperti ini:

```text
Database: ds_sql_learning
│
├── customers   (customer_id [PK], name, email, city, ...)
├── categories  (category_id [PK], category_name)
├── products    (product_id [PK], name, price, category_id [FK → categories])
└── orders      (order_id [PK], customer_id [FK → customers], order_date, ...)

* customers dan orders berelasi one-to-many: satu customer bisa punya banyak order, tapi satu order hanya milik satu customer. Relasi ini terbentuk lewat customer_id (PK di customers, FK di orders).
* products dan categories juga one-to-many: satu kategori bisa punya banyak produk, lewat category_id (PK di categories, FK di products).
* Perhatikan: tabel orders tidak menyimpan nama atau email customer secara langsung — ia cukup menyimpan customer_id, lalu detail customer diambil dengan menghubungkan ke tabel customers saat dibutuhkan. Ini prinsip dasar relational design: hindari duplikasi data.

# 3. SELECT

## 3.1 Anatomi Dasar Query SELECT

Bentuk paling sederhana

```SQL
SELECT column_list
FROM table_name;

Beberapa hal fundamental
* Keyword tidak case sensitive (SELECT = select)
* Semicolon (;) menandai akhir statement
* Urutan eksekusi SQL: `FROM` → `SELECT`.
    * `FROM` menentukan tabel sumber, lalu `SELECT` memilih/menghitung kolom.
    * Konsep ini penting untuk memahami `WHERE` dan `GROUP BY` nantinya.


## 3.2 SELECT * vs Kolom Eksplisit

```sql
SELECT * FROM customers;

" * " = semua kolom.

penggunaan * dihindari karena:
* Kalau schema tabel berubah (ada kolom baru ditambahkan), hasil query ikut berubah tanpa Anda sadari — ini bisa mematahkan pipeline/dashboard yang bergantung pada urutan atau jumlah kolom tertentu.
* Menarik kolom yang tidak dibutuhkan membebani transfer data, apalagi di tabel dengan kolom besar (misal kolom teks panjang atau JSON).
* Kolom eksplisit membuat intent query jelas dibaca orang lain (atau diri Anda sendiri enam bulan kemudian).

Bandingkan dengan versi eksplisit

```sql

SELECT
    customer_id,
    name,
    email
FROM customers;
```

Urutan kolom di hasil mengikuti urutan yang Anda tulis di SELECT, bukan urutan asli di tabel.

## 3.3 Alias Kolom (AS)

Alias mengganti nama tampilan kolom di hasil query, tanpa mengubah nama asli di database:

```sql
SELECT 
    name AS customers_name,
    email AS contact_email,
FROM customers;
```

* AS > opsional (name customer_name juga valid), ditulis untuk keterbacaan
* Kalau alias mengandung spasi atau kata yang bentrok dengan reserved word, PostgreSQL butuh tanda kutip ganda: SELECT price AS "Price (IDR)"
* Alias berguna terutama nanti saat hasil ekspresi/agregasi butuh nama yang lebih deskriptif daripada sum, count, dsb

## 3.4 Expression dalam SELECT

SELECT tidak cuma menarik kolom mentah, tapi bisa menghitung kolom baru on the fly, mirip df.assign() di pandas:

```sql
SELECT 
    name,
    price * 1.11 AS price_with_tax
FROM product;

Atau menggabungkan string (PostgreSQL pakai ||, bukan + seperti sebagian bahasa lain):

```sql
SELECT
    name || ' - ' || email AS contact_info
FROM customers;

## 3.5 Komentar

```sql
-- komentar satu baris
SELECT * FROM customers;

/* komentar
   multi baris */

## Latihan

* Tampilkan semua kolom dari tabel customers.
*  Tampilkan hanya name dan email dari customers.
* Tampilkan name dan price dari tabel products, dengan alias product_name dan product_price.
* Buat kolom hitungan baru: tampilkan name, price, dan price dikurangi diskon 10% (beri alias discounted_price).

# 4. WHERE — Memfilter Data

## 4.1 Apa yang Sebenarnya Dilakukan WHERE

WHERE menguji setiap baris satu persatu : kondisi di dalamnya dievaluasi menjadi TRUE atau FALSE untuk tiap baris, dan hanya baris yang menghasilkan TRUE yang diteruskan ke SELECT.

* Ini persis konsep dari pandas: df[df['price'] > 100000] — boolean indexing. WHERE adalah versi SQL dari itu.

*  Urutan eksekusi logis SQL: `FROM → WHERE → SELECT`
* FROM: Database menentukan tabel sumber yang akan digunakan.
* WHERE: Database menyaring baris berdasarkan kondisi yang diberikan.
* SELECT: Setelah baris tersaring, database memilih atau menghitung kolom yang akan ditampilkan.
* Penting: Alias yang didefinisikan di `SELECT` belum tersedia ketika `WHERE` diproses.
* Karena itu, pada level dasar Anda tidak bisa menggunakan alias dari `SELECT` langsung di `WHERE`.
* Contoh:

  ```sql
  SELECT
      price * 0.9 AS discounted_price
  FROM products
  WHERE discounted_price < 100;
  ```

  Tidak valid pada SQL umum karena `WHERE` diproses sebelum `SELECT`.
* Konsep ini penting karena akan membantu memahami urutan eksekusi SQL, terutama ketika nanti mempelajari `GROUP BY`, `HAVING`, `ORDER BY`, CTE, dan window function.


Bentuk umum:

```sql
SELECT column_list
FROM table_name,
WHERE condition;

## 4.2 Operator Perbandingan

```text
=     sama dengan
!=    tidak sama dengan   (setara dengan <>)
<>    tidak sama dengan   (standar ANSI SQL, lebih portable)
>     lebih besar
<     lebih kecil
>=    lebih besar atau sama dengan
<=    lebih kecil atau sama dengan

tipe data menentukan cara penulisan nilai

* Angka: tidak perlu tanda kutip → price > 100000
* Teks: wajib tanda kutip satu → city = 'Yogyakarta'
* Tanggal: wajib tanda kutip, format 'YYYY-MM-DD' → order_date >= '2025-06-01'

## 4.3 Operator Logika: AND, OR, NOT

* AND → kedua kondisi harus TRUE
* OR → salah satu kondisi cukup TRUE
* NOT → membalik hasil kondisi

precedence (urutan pengerjaan): NOT dikerjakan lebih dahulu, lalu AND baru OR

Contoh Jebakan:

``` sql
-- Maksud: customer dari Yogyakarta ATAU Jakarta, yang keduanya harus terdaftar setelah 2025-03-01
SELECT *
FROM customers
WHERE city = 'Yogyakarta' OR city = 'Jakarta' AND created_at >= '2025-03-01';

Karena AND dikerjakan lebih dulu daripada OR, query di atas sebenarnya berarti: "city = Yogyakarta, ATAU (city = Jakarta DAN created_at >= 2025-03-01)" — customer Yogyakarta manapun akan lolos tanpa syarat tanggal. Ini bug diam-diam, tidak error tapi hasilnya salah. Perbaikannya, pakai tanda kurung eksplisit:

```sql
SELECT *
FROM customers
WHERE (city = 'Yogyakarta' OR city = 'Jakarta') AND created_at >= '2025-03-01';

Aturan praktis: begitu Anda mencampur AND dan OR dalam satu WHERE, selalu tambahkan tanda kurung eksplisit — jangan andalkan hafalan precedence.

## 4.4 IN — Uji Keanggotaan

IN adalah bentuk singkat untuk banyak kondisi OR yang membandingkan kolom sama:

```sql
-- Ini:
WHERE city = 'Yogyakarta' OR city = 'Malang' OR city = 'Bandung'

-- Sama persis dengan ini, lebih ringkas dan lebih terbaca:
WHERE city in ('Yogyakarta', 'Jakarta', 'Bandung')


## 4.5 BETWEEN — Rentang Nilai

```sql
WHERE price BETWEEN 100000 AND 300000;

Catatan penting: BETWEEN inclusive di kedua ujung — nilai persis 100000 atau 300000 ikut lolos. Ini setara dengan:

```SQL
WHERE price >= 100000 AND price <= 300000

Berlaku juga untuk tanggal: order_date BETWEEN '2025-06-01' AND '2025-06-30'

## 4.6 LIKE — Pencocokan Pola Teks

Untuk mencari teks berdasarkan pola, bukan kecocokan persis. Dua wildcard:

* % → mewakili sembarang jumlah karakter (termasuk nol karakter)
* _ → mewakili tepat satu karakter

```sql
WHERE name LIKE 'A%'      -- nama diawali huruf A
WHERE email LIKE '%@mail.com'  -- email diakhiri @mail.com
WHERE name LIKE '_i%'     -- huruf kedua adalah 'i'

Catatan: LIKE di PostgreSQL case-sensitive secara default ('andi' ≠ 'Andi'). Kalau butuh pencarian tanpa peduli huruf besar/kecil, PostgreSQL punya ILIKE:

```sql
WHERE name ILIKE 'andi%'  -- cocok dengan 'Andi', 'ANDI', 'andi', dst

## 4.7 Catatan Penting: WHERE dan NULL

* `WHERE phone = NULL` tidak akan mengembalikan baris apa pun, meskipun `phone` memiliki nilai `NULL`.
* Ini bukan bug, tetapi perilaku SQL karena `NULL` tidak dapat dibandingkan menggunakan `=`.
* Untuk memfilter `NULL`, gunakan `IS NULL`.
* Pembahasan lengkap tentang `NULL` ada di Step 6.
* Ingat hal ini saat memfilter kolom seperti `phone` atau `city` yang dapat berisi `NULL`.


## Implementasi

```sql
-- Produk dengan harga di atas 100000
SELECT name, price
FROM products
WHERE price > 100000;

-- Customer dari Yogyakarta
SELECT name, city
FROM customers
WHERE city = 'Yogyakarta';

-- Order dengan status completed ATAU pending (pakai IN)
SELECT order_id, status, total_amount
FROM orders
WHERE status IN ('completed', 'pending');

-- Produk dengan harga antara 100000 dan 300000 (pakai BETWEEN)
SELECT name, price
FROM products
WHERE price BETWEEN 100000 AND 300000;

-- Customer yang namanya diawali huruf 'A' atau 'B' (pakai LIKE + OR, lalu bandingkan dengan cara lain)
SELECT name
FROM customers
WHERE name LIKE 'A%' OR name LIKE 'B%';

-- Order antara tanggal tertentu, DAN statusnya bukan cancelled
SELECT order_id, order_date, status
FROM orders
WHERE order_date BETWEEN '2025-06-01' AND '2025-07-31'
  AND status != 'cancelled';

## Latihan

1. Tampilkan semua produk dengan harga lebih dari 100000.
2. Tampilkan semua customer dari kota Yogyakarta.
3. Tampilkan order yang terjadi antara tanggal tertentu (pilih sendiri rentangnya) menggunakan BETWEEN.
4. Tampilkan produk dengan kategori category_id 1 atau 4, menggunakan IN.
5. Tampilkan customer yang emailnya mengandung domain mail.com menggunakan LIKE.
6. Latihan precedence: tampilkan order yang statusnya completed dan (total_amount di atas 300000 atau order_date setelah 2025-07-01) — perhatikan baik-baik di mana Anda taruh tanda kurung.

# 5. ORDER BY, LIMIT, DISTINCT

* `ORDER BY` digunakan untuk mengurutkan baris hasil query berdasarkan satu atau lebih kolom.

* Tanpa `ORDER BY`:

  * SQL tidak menjamin urutan baris yang dikembalikan.
  * Hasil mungkin terlihat mengikuti urutan `INSERT` atau penyimpanan fisik.
  * Namun, database bebas mengembalikan baris dalam urutan lain jika dianggap lebih efisien.
  * Urutan yang terlihat konsisten bukan berarti urutan tersebut dijamin.

* Prinsip penting:

  * Jika urutan hasil penting, selalu gunakan `ORDER BY`.
  * Jangan mengandalkan urutan hasil hanya karena selama ini terlihat konsisten.

* Urutan dasar:

```sql
SELECT *
FROM products
ORDER BY price;
```

* Secara default, `ORDER BY` menggunakan urutan ascending (`ASC`):

  * angka: kecil → besar
  * teks: A → Z
  * tanggal: lama → baru

* Untuk urutan descending (`DESC`):

```sql
SELECT *
FROM products
ORDER BY price DESC;
```

* Hasilnya:

  * harga tertinggi → terendah.

* Dapat mengurutkan berdasarkan beberapa kolom:

```sql
SELECT *
FROM customers
ORDER BY city ASC, name ASC;
```

* Database akan:

  1. Mengurutkan berdasarkan `city`.
  2. Jika beberapa customer memiliki `city` yang sama, mengurutkannya berdasarkan `name`.

* Konsep utama:

```text
Tanpa ORDER BY
→ urutan tidak dijamin

Dengan ORDER BY
→ urutan ditentukan secara eksplisit
```

* Prinsip praktis untuk dunia kerja:

  * Jangan menganggap urutan `SELECT *` sebagai urutan data yang sebenarnya.
  * Jika laporan, ranking, top-N, atau hasil analisis membutuhkan urutan tertentu, selalu tuliskan `ORDER BY`.


```sql
SELECT name, price
FROM products
ORDER BY price;          -- default: ASC (ascending, kecil ke besar)

SELECT name, price
FROM products
ORDER BY price DESC;     -- descending, besar ke kecil

Bisa mengurutkan berdasarkan lebih dari satu kolom — kolom pertama jadi prioritas utama, kolom berikutnya jadi tie-breaker (penentu urutan kalau nilai kolom pertama sama):

```sql
SELECT name, category_id, price
FROM products
ORDER BY category_id ASC, price DESC;
-- urutkan per kategori dulu, lalu dalam satu kategori yang sama, harga termahal duluan

Bisa juga mengurutkan pakai expression atau posisi kolom (angka mengacu urutan kolom di SELECT), tapi ini kurang direkomendasikan untuk query yang disimpan karena kalau urutan kolom SELECT berubah, logikanya ikut berubah tanpa disadari:

```sql
ORDER BY 3 DESC   -- mengacu kolom ke-3 di SELECT, hindari untuk query permanen

## 5.2 LIMIT — Membatasi Jumlah Baris

* LIMIT memotong hasil menjadi sejumlah baris tertentu, dihitung setelah ORDER BY diterapkan (kalau ada).
* penting dipahami: urutan eksekusi logis sekarang jadi FROM → WHERE → ORDER BY → LIMIT.

```sql
SELECT name, price
FROM products
ORDER BY price DESC
LIMIT 5;   -- 5 produk termahal

* Jebakan umum yang perlu diperhatikan adalah penggunaan `LIMIT` tanpa `ORDER BY`. Query tersebut hanya mengembalikan sejumlah baris dari hasil query tanpa menjamin bahwa baris yang diperoleh merupakan baris dengan nilai tertentu.

* Sebagai contoh, `LIMIT 5` tanpa `ORDER BY` hanya berarti mengambil 5 baris dari hasil query. Hasil tersebut tidak dapat dianggap sebagai 5 produk dengan harga tertinggi.

* Untuk memperoleh hasil Top-N atau Bottom-N secara konsisten, `ORDER BY` dan `LIMIT` perlu digunakan secara bersamaan.

* Pola umum:

  * Top-N → `ORDER BY ... DESC` + `LIMIT`
  * Bottom-N → `ORDER BY ... ASC` + `LIMIT`

* Contoh Top-5 produk berdasarkan harga:

```sql
SELECT *
FROM products
ORDER BY price DESC
LIMIT 5;
```

* Urutan logisnya:

  1. `ORDER BY` menentukan urutan data.
  2. `LIMIT` mengambil sejumlah baris dari urutan tersebut.

* Dengan demikian, kombinasi `ORDER BY` dan `LIMIT` merupakan pola dasar yang penting dalam pengambilan data Top-N dan Bottom-N.


Ada juga OFFSET, untuk melompati sejumlah baris pertama — berguna untuk pagination:
```sql
SELECT name, price
FROM products
ORDER BY price DESC
LIMIT 5 OFFSET 5;   -- baris ke-6 sampai ke-10 (halaman kedua, page size 5)

## 5.3 DISTINCT — Menghilangkan Duplikat

DISTINCT mengembalikan hanya kombinasi nilai yang unik dari kolom yang dipilih.

```sql
SELECT DISTINCT city
FROM customers;

Poin fundamental: DISTINCT beroperasi pada kombinasi seluruh kolom yang di-SELECT, bukan per kolom secara terpisah.

```sql
SELECT DISTINCT city, status  -- ini contoh hipotetis, city ada di customers bukan orders

Contoh

```sql
-- Kombinasi unik category_id + harga pembulatan ratusan ribu
SELECT DISTINCT category_id
FROM products;
-- hasil: daftar category_id unik saja (misal 1,2,3,4,5)

SELECT DISTINCT category_id, stock
FROM products;
-- hasil: setiap kombinasi (category_id, stock) yang berbeda dianggap baris unik tersendiri,
-- meskipun category_id-nya sama dengan baris lain

Kalau Anda hanya butuh nilai unik dari satu kolom, SELECT hanya kolom itu saja — begitu ada kolom kedua, definisi "unik"-nya ikut berubah mencakup kombinasi keduanya.

## 5.4 Kombinasi Ketiganya

```sql
SELECT DISTINCT city
FROM customers
ORDER BY city;

SELECT name, price
FROM products
WHERE category_id = 1
ORDER BY price DESC
LIMIT 3;

Urutan penulisan clause dalam query wajib: SELECT ... FROM ... WHERE ... ORDER BY ... LIMIT ... — ini bukan pilihan gaya, tapi syntax yang harus diikuti persis urutan ini.

## Implementasi

```sql
-- 10 produk termahal
SELECT name, price
FROM products
ORDER BY price DESC
LIMIT 10;

-- 10 order dengan total_amount terbesar
SELECT order_id, total_amount, order_date
FROM orders
ORDER BY total_amount DESC
LIMIT 10;

-- Daftar kota unik tempat customer berasal
SELECT DISTINCT city
FROM customers
ORDER BY city;

-- Daftar category_id unik di tabel products
SELECT DISTINCT category_id
FROM products
ORDER BY category_id;

-- 5 produk termurah dalam kategori tertentu
SELECT name, price
FROM products
WHERE category_id = 2
ORDER BY price ASC
LIMIT 5;

## Latihan

* Tampilkan 10 produk termahal (nama dan harga saja).
* Tampilkan 10 transaksi (order) terbesar berdasarkan total_amount.
* Tampilkan daftar kota unik dari tabel customers.
* Tampilkan daftar status order yang unik dari tabel orders.
* Tampilkan 5 customer pertama yang terdaftar (created_at paling awal).
* Latihan pagination: tampilkan produk urutan ke-6 sampai ke-10 berdasarkan harga tertinggi (pakai LIMIT + OFFSET).
* Latihan gabungan: dari kategori dengan category_id = 3, tampilkan 3 produk dengan stock terbanyak.

## 6. NULL — Merepresentasikan Data yang Tidak Diketahui

## 6.1 Apa Itu NULL Sebenarnya

NULL bukan nilai. NULL adalah penanda bahwa nilai tersebut tidak diketahui / tidak ada / belum diisi.

* `NULL` merupakan konsep dasar SQL yang penting untuk memahami pengolahan data.

* `NULL` berarti data atau nilai tersebut tidak tersedia atau tidak diketahui.

* `NULL` berbeda dengan:

  * String kosong (`''`) → nilai diketahui berupa tidak ada karakter.
  * `0` → nilai diketahui berupa angka nol.
  * `NULL` → tidak ada nilai yang diketahui.

* Pada tabel `customers`, beberapa customer sengaja memiliki `phone = NULL`.

* Contoh:

  ```text
  phone = ''  → nomor telepon diketahui kosong
  phone = 0   → nilai yang tersimpan adalah 0
  phone = NULL → nomor telepon tidak tersedia/tidak diketahui
  ```

* Perbedaan ini penting karena SQL memperlakukan `NULL` secara khusus dalam operasi perbandingan dan filtering.


## 6.2 Three-Valued Logic: TRUE, FALSE, UNKNOWN

* SQL menggunakan three-valued logic, yaitu kondisi dapat menghasilkan:

  * `TRUE`
  * `FALSE`
  * `UNKNOWN`

* Ketika `NULL` terlibat dalam perbandingan seperti `=`, `!=`, `>`, `<`, dan lainnya, hasilnya adalah `UNKNOWN`.

* Contoh:

  ```sql
  NULL = NULL
  ```

  menghasilkan `UNKNOWN`, bukan `TRUE`, karena nilai `NULL` tidak diketahui.

* `NULL` dapat dianggap sebagai ketiadaan informasi, sehingga database tidak dapat memastikan hasil perbandingan.

* Dalam `WHERE`, hanya kondisi yang bernilai `TRUE` yang akan menghasilkan baris.

  * `TRUE` → baris ditampilkan
  * `FALSE` → baris tidak ditampilkan
  * `UNKNOWN` → baris tidak ditampilkan

* Konsep ini menjelaskan mengapa:

  ```sql
  WHERE phone = NULL
  ```

  tidak menghasilkan baris.

* Pemahaman `TRUE`, `FALSE`, dan `UNKNOWN` menjadi penting ketika `NULL` digunakan bersama `AND` dan `OR`.


## 6.3 IS NULL dan IS NOT NULL

* Operator perbandingan seperti `=` dan `!=` tidak dapat digunakan untuk mendeteksi `NULL`.

* Ketika `NULL` terlibat dalam perbandingan, hasilnya adalah `UNKNOWN`, bukan `TRUE`.

* Oleh karena itu, kondisi berikut tidak dapat digunakan:

  ```sql
  WHERE phone = NULL
  ```

* SQL menyediakan operator khusus untuk memeriksa keberadaan `NULL`:

  * `IS NULL` → memeriksa apakah nilai adalah `NULL`.
  * `IS NOT NULL` → memeriksa apakah nilai bukan `NULL`.

* Contoh:

  ```sql
  SELECT *
  FROM customers
  WHERE phone IS NULL;
  ```

* Untuk mencari data yang memiliki nilai:

  ```sql
  SELECT *
  FROM customers
  WHERE phone IS NOT NULL;
  ```

* Prinsip utama:

  ```text
  = NULL       → salah untuk mendeteksi NULL
  IS NULL      → digunakan untuk mendeteksi NULL
  IS NOT NULL  → digunakan untuk mendeteksi nilai yang bukan NULL
  ```


```sql
SELECT name, phone
FROM customers
WHERE phone IS NULL;
-- akan menampilkan: Budi Santoso, Eko Wijaya, Indra Saputra, Maya Anggraini

SELECT name, phone
FROM customers
WHERE phone IS NOT NULL;
-- kebalikannya: semua customer yang phone-nya terisi

Inilah jawaban atas kesalahan klasik yang jadi goal Step ini:

```sql
WHERE phone = NULL       -- SALAH: selalu UNKNOWN, tidak pernah meloloskan baris apapun
WHERE phone IS NULL      -- BENAR

## 6.4 NULL Bertemu AND / OR

### 6.4 NULL Bertemu AND / OR

* `NULL` menghasilkan `UNKNOWN` ketika digunakan dalam perbandingan.

* Ketika `UNKNOWN` digabungkan dengan `AND` atau `OR`, hasilnya mengikuti logika tiga nilai SQL.

* `AND`:

  * `TRUE AND UNKNOWN` → `UNKNOWN`
  * `FALSE AND UNKNOWN` → `FALSE`
  * `UNKNOWN AND UNKNOWN` → `UNKNOWN`
  * Jika salah satu kondisi pasti `FALSE`, keseluruhan kondisi pasti `FALSE`.

* `OR`:

  * `TRUE OR UNKNOWN` → `TRUE`
  * `FALSE OR UNKNOWN` → `UNKNOWN`
  * `UNKNOWN OR UNKNOWN` → `UNKNOWN`
  * Jika salah satu kondisi pasti `TRUE`, keseluruhan kondisi pasti `TRUE`.

* Cara mengingat secara intuitif:

  * `AND` membutuhkan semua kondisi `TRUE`. Jika salah satunya `FALSE`, hasil pasti `FALSE`.
  * `OR` hanya membutuhkan satu kondisi `TRUE`. Jika salah satunya `TRUE`, hasil pasti `TRUE`.
  * `UNKNOWN` berarti database belum dapat menentukan hasil akhirnya.

* Dalam `WHERE`, hasil akhirnya tetap mengikuti aturan:

  * `TRUE` → baris ditampilkan.
  * `FALSE` → baris tidak ditampilkan.
  * `UNKNOWN` → baris tidak ditampilkan.

* Contoh:

```sql
SELECT *
FROM customers
WHERE phone IS NULL
  AND city = 'Jakarta';
```

* Artinya:

  * Customer harus memiliki `phone` yang `NULL`.
  * Customer juga harus berasal dari Jakarta.
  * Kedua kondisi harus terpenuhi.

* Pemahaman `UNKNOWN` bersama `AND` dan `OR` penting karena menjelaskan mengapa query yang melibatkan `NULL` terkadang menghasilkan jumlah baris yang tidak sesuai dengan intuisi awal.


## 6.5 NULL vs 0 vs '' (String Kosong)

| Nilai  | Arti                                                | Contoh                                           |
| ------ | --------------------------------------------------- | ------------------------------------------------ |
| `NULL` | Tidak ada nilai atau nilai tidak diketahui          | `phone = NULL` → nomor telepon belum tersedia    |
| `0`    | Nilai numerik yang diketahui, yaitu angka nol       | `stock = 0` → produk memang tidak memiliki stok  |
| `''`   | Nilai teks yang diketahui dengan panjang 0 karakter | `name = ''` → terdapat isian teks, tetapi kosong |


* `''` dan `NULL` merupakan dua nilai yang berbeda dalam SQL.
* `WHERE phone IS NULL` hanya akan menemukan nilai `NULL`.
* Data dengan `phone = ''` tidak akan ikut terdeteksi karena `''` merupakan string yang valid.
* Kesalahan seperti ini dapat terjadi ketika:

  * Form input tidak melakukan validasi.
  * Data hasil scraping menggunakan string kosong sebagai pengganti `NULL`.
  * Proses ETL tidak melakukan standardisasi missing value.
* Dampaknya adalah hasil analisis missing value menjadi tidak akurat.
* Sebagai Data Scientist, hal ini perlu diperhatikan sebagai data quality issue, bukan sekadar masalah syntax SQL.
* Salah satu langkah pemeriksaan dapat dilakukan dengan:

```sql
SELECT *
FROM customers
WHERE phone IS NULL
   OR phone = '';
```

* Prinsip penting: pahami perbedaan antara `NULL`, string kosong (`''`), dan nilai seperti `0` sebelum melakukan analisis atau filtering data.


## 6.6 Sekilas: Dampak NULL pada Fungsi Agregat

* `NULL` diperlakukan secara khusus oleh fungsi agregat seperti `COUNT()`, `SUM()`, dan `AVG()`.

* Perbedaan penting:

  * `COUNT(*)` → menghitung semua baris, termasuk baris yang memiliki `NULL`.
  * `COUNT(nama_kolom)` → hanya menghitung baris yang memiliki nilai pada kolom tersebut; `NULL` diabaikan.

* Contoh:

  ```sql
  SELECT COUNT(*) FROM customers;
  ```

  → menghitung seluruh customer.

* Sedangkan:

  ```sql
  SELECT COUNT(phone) FROM customers;
  ```

  → hanya menghitung customer yang memiliki nilai `phone`.

* Prinsip yang perlu diingat:

  * `COUNT(*)` → menghitung baris.
  * `COUNT(kolom)` → menghitung nilai yang bukan `NULL` pada kolom.

* Dampak `NULL` pada fungsi agregat lainnya akan dibahas lebih lengkap pada Module 02, Step 2.

* Untuk saat ini, cukup pahami bahwa `NULL` dapat membuat sebagian data tidak ikut dihitung dalam fungsi agregat.


## 6.7 COALESCE — Mengganti NULL dengan Nilai Default

* `COALESCE()` digunakan untuk mengganti `NULL` dengan nilai default atau alternatif.

* `COALESCE()` mengembalikan nilai pertama yang tidak `NULL` dari daftar argumen.

* Sintaks dasar:

```sql
COALESCE(nilai1, nilai2, nilai3, ...)
```

* Contoh:

```sql
SELECT
    name,
    COALESCE(phone, 'Tidak tersedia') AS phone
FROM customers;
```

* Jika `phone` memiliki nilai:

  * Nilai `phone` ditampilkan.

* Jika `phone` adalah `NULL`:

  * `'Tidak tersedia'` ditampilkan.

* Contoh lain:

```sql
SELECT
    name,
    COALESCE(city, 'Kota tidak diketahui') AS city
FROM customers;
```

* `COALESCE()` tidak mengubah data asli di database.

* Nilai `NULL` hanya diganti pada hasil query.

* Prinsip utama:

```text
COALESCE(nilai, nilai_default)
        ↓
Jika nilai tidak NULL → gunakan nilai
Jika nilai NULL       → gunakan nilai_default
```

* `COALESCE()` sangat berguna dalam:

  * Menampilkan data yang lebih mudah dibaca.
  * Menangani missing value dalam hasil query.
  * Membuat laporan dan output analisis lebih informatif.

* Untuk tahap ini, cukup pahami pola dasar:

```sql
COALESCE(column, default_value)
```


Pertanyaan bagus — ini extension yang pas untuk soal #5. COALESCE bisa menerima lebih dari satu kolom sekaligus, tapi ada dua pola berbeda yang sering tertukar, tergantung apa yang ingin dicapai.

**Pola 1: COALESCE terpisah per kolom** (kemungkinan besar ini yang Anda maksud)

`city` dan `phone` itu dua atribut yang maknanya berbeda, jadi masing-masing butuh default text sendiri-sendiri — bukan saling menggantikan:

```sql
SELECT
    customer_id,
    name,
    COALESCE(city, 'Kota tidak diketahui') AS city,
    COALESCE(phone, 'Tidak ada nomor') AS phone
FROM customers
WHERE city IS NULL OR phone IS NULL;
```

Poin penting: `WHERE` tetap pakai `city IS NULL OR phone IS NULL` — **bukan** `COALESCE(city, ...) IS NULL`. COALESCE hanya mengubah tampilan di hasil `SELECT`, bukan data aslinya. Kalau COALESCE ditaruh di `WHERE`, hasilnya justru tidak akan pernah NULL (karena sudah tergantikan default), jadi `IS NULL` tidak akan pernah cocok — filter jadi tidak berfungsi.

**Pola 2: COALESCE lintas kolom** (satu kolom jadi fallback kolom lain)

Ini pola berbeda — dipakai kalau dua kolom punya *makna yang bisa saling menggantikan*, bukan sekadar dua atribut berbeda. Kurang cocok untuk `city`/`phone`, tapi contoh use case-nya kalau seandainya ada `preferred_contact` dan `backup_contact`:

```sql
SELECT
    customer_id,
    COALESCE(preferred_contact, backup_contact, 'Tidak ada kontak') AS contact
FROM customers;
```

COALESCE cek argumen pertama dulu — kalau NULL, lanjut ke argumen berikutnya — begitu seterusnya sampai ketemu yang bukan NULL. Bisa menerima argumen sebanyak apapun, bukan cuma dua.

Untuk soal #5, query `WHERE` Anda sudah tepat — logikanya benar (`IS NULL` + `OR`, sama seperti yang dibahas di 6.4). Menambahkan COALESCE di `SELECT` hanya soal presentasi hasil, tidak mengubah filter-nya.

Lanjutkan lima soal lainnya kapan siap.

## Implementasi

```sql
-- Customer yang belum mengisi nomor telepon
SELECT name, phone
FROM customers
WHERE phone IS NULL;

-- Customer yang sudah mengisi nomor telepon
SELECT name, phone
FROM customers
WHERE phone IS NOT NULL;

-- Buktikan sendiri: query ini TIDAK akan mengembalikan baris apapun
SELECT name, phone
FROM customers
WHERE phone = NULL;

-- Customer yang city ATAU phone-nya belum diisi (pakai OR)
SELECT name, city, phone
FROM customers
WHERE city IS NULL OR phone IS NULL;

-- Tampilkan dengan nilai pengganti yang lebih mudah dibaca
SELECT name, COALESCE(city, 'Kota tidak diketahui') AS city
FROM customers;

Latihan

Kerjakan tanpa melihat contoh di atas:

1. Tampilkan semua customer yang kolom city-nya NULL.
2. Tampilkan semua customer yang kolom city-nya tidak NULL, diurutkan berdasarkan created_at terbaru.
3. Buktikan sendiri (jalankan dan amati hasilnya): apa yang terjadi kalau Anda menjalankan WHERE city != NULL? Bandingkan dengan WHERE city IS NOT NULL.
4. Tampilkan customer yang phone-nya NULL dan kotanya Yogyakarta — perhatikan apakah hasilnya sesuai dugaan Anda berdasarkan pemahaman 6.4.
5. Pakai COALESCE untuk menampilkan name dan phone, di mana phone yang NULL diganti teks 'Belum ada data'.

# 7. Fundamental Challenge

7.1 Aturan Main

* Kerjakan langsung di DBeaver, database ds_sql_learning.
* Jangan buka scrollback percakapan ini atau catatan Step 1–6 sambil mengerjakan. Kalau tiba-tiba lupa syntax di tengah jalan, itu sinyal bagian mana yang perlu diulang — bukan alasan untuk mengintip contoh.
* Terjemahkan soal ke pertanyaan bisnis dulu di kepala Anda ("saya butuh data apa, disaring bagaimana, diurutkan bagaimana"), baru tulis SQL-nya — bukan mencocokkan soal ke template yang pernah dilihat.

7.2 Soal

Tanpa referensi, jawab keenam ini:

1. Tampilkan 10 customer yang paling baru terdaftar.
2. Tampilkan 10 produk termahal.
3. Tampilkan produk dengan harga di antara dua nilai pilihan Anda sendiri.
4. Tampilkan customer dari satu kota pilihan Anda sendiri.
5. Tampilkan data (tabel dan kolom bebas Anda pilih) yang punya nilai NULL.
6. Tampilkan nilai unik dari satu kolom pilihan Anda sendiri.